#Connect With Drive

In [ ]:
from google.colab import drive
drive.mount("/gdrive")

Mounted at /gdrive


# Libraries Installation

In [ ]:
!pip install -qU datasets==3.2.0 optimum==1.24.0
!pip install -qU openai==1.61.0 wandb
!pip install -qU json-repair==0.29.1
!pip install PyPDF2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.6/433.6 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 126.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install transformers -U

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.8/558.8 kB 21.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.33.4
    Uninstalling huggingface-hub-0.33.4:
      Successfully uninstalled huggingface-hub-0.33.4
  Attempting uninstall: transformers
    Found existing installation: transformers 4.53.3
    Uninstalling transformers-4.53.3:
      Successfully uninstalled transformers-4.53.3


In [ ]:
!pip install bitsandbytes -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 8.1 MB/s eta 0:00:00


# Getting Secret Keys

In [ ]:
from google.colab import userdata
import wandb

huggingface_key = userdata.get('huggingface')
wandb_key = userdata.get('wandb')
openai_key = userdata.get('openai')

wandb.login(key=wandb_key)
!huggingface-cli login --token {huggingface_key}


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: christeenhallak33 (christeenhallak33-coretech-mena) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `huggingfaceToken` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `huggingfaceToken`


# Import Libraries

In [ ]:
import json
import os
from os.path import join
import random
from tqdm.auto import tqdm
import requests

from pydantic import BaseModel, Field
from typing import List, Optional, Literal
from datetime import datetime

import json_repair

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

data_dir = '/content/gdrive/MyDrive/Artifical Intelligence/LLM/Fine Tunning/question_dataset_1000.jsonl'
saved_dir = '/content/gdrive/MyDrive/Artificial Intelligence/LLM/Fine Tunning/'
gbase_model_id = "Qwen/Qwen2.5-1.5B-Instruct"

device = "cuda"
torch_dtype = None

def parse_json(text):
    try:
        return json_repair.loads(text)
    except:
        return None

ModuleNotFoundError: No module named 'json_repair'

# Upload File From Local

In [ ]:

# Step 1: Upload file from your local device
from google.colab import files
uploaded = files.upload()  # Opens file picker

# Step 2: Extract the file path (assumes single file uploaded)
import os

pdf_path = next(iter(uploaded))  # Get the uploaded filename


Saving history of Oman.pdf to history of Oman.pdf


# Extract Text From PDF

In [ ]:
# Step 3: Use your existing function (adjusted to skip URL handling)
import PyPDF2

def extract_text_from_pdf_local(pdf_path: str) -> str:
    """Extract text from a local PDF file."""
    try:
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            pages = reader.pages
            pdf_text = " ".join(page.extract_text() or "" for page in pages).strip()
            return pdf_text
    except Exception as e:
        return f"Error reading PDF: {e}"

# Step 4: Extract text from the uploaded PDF
pdf_content_local = extract_text_from_pdf_local(pdf_path)

pdf_content = ' '.join(pdf_content_local.split())
print(pdf_content)

التراث والتاريخ العريق في سلطنة عُمان مقدمة سلطنة عُمان، الواقعة في الركن الجنوبي الشرقي من شبه الجزيرة العربية، تُعد واحدة من أقدم الدول التي استوطنتها الحضارات البشرية، ولعبت دورًا مهمًا في تاريخ المنطقة والعالم. بفضل موقعها الجغرافي الفريد، المُطل على ممرات بحرية حيوية، أصبحت مركزًا للتجارة والمالحة منذ آالف السنين. وال يقتصر تميز عُمان على موقعها فقط، بل يمتد إلى تراثها الغني الذي يجمع بين عبق التاريخ وروح األصالة. عُمان عبر العصور عُمان القديمة يعود تاريخ االستيطان البشري في عمان إلى ما قبل أكثر من 10,000 سنة، حيث دلت الحفريات األثرية على وجود حضارات قديمة مثل حضارة "مجان"، التي ذكرتها النصوص السومرية وكانت تشتهر بتجارة النحاس والديوريت. العصور اإلسالمية مع بداية القرن السابع الميالدي، دخل اإلسالم إلى عُمان طوعًا، وكان العمانيون من أوائل من اعتنقوا اإلسالم دون قتال. وقد ساهمت عُمان في نشر اإلسالم في شرق أفريقيا والهند وجنوب شرق آسيا عن طريق تجارها وبحّارتها. الدولة اليعربية والنهضة البحرية في القرن السابع عشر، برزت الدولة اليعربية كقوة بحرية عُمانية استطاعت طرد البرتغاليين من سواح

In [ ]:
pdf_content

'التراث والتاريخ العريق في سلطنة عُمان مقدمة سلطنة عُمان، الواقعة في الركن الجنوبي الشرقي من شبه الجزيرة العربية، تُعد واحدة من أقدم الدول التي استوطنتها الحضارات البشرية، ولعبت دورًا مهمًا في تاريخ المنطقة والعالم. بفضل موقعها الجغرافي الفريد، المُطل على ممرات بحرية حيوية، أصبحت مركزًا للتجارة والمالحة منذ آالف السنين. وال يقتصر تميز عُمان على موقعها فقط، بل يمتد إلى تراثها الغني الذي يجمع بين عبق التاريخ وروح األصالة. عُمان عبر العصور عُمان القديمة يعود تاريخ االستيطان البشري في عمان إلى ما قبل أكثر من 10,000 سنة، حيث دلت الحفريات األثرية على وجود حضارات قديمة مثل حضارة "مجان"، التي ذكرتها النصوص السومرية وكانت تشتهر بتجارة النحاس والديوريت. العصور اإلسالمية مع بداية القرن السابع الميالدي، دخل اإلسالم إلى عُمان طوعًا، وكان العمانيون من أوائل من اعتنقوا اإلسالم دون قتال. وقد ساهمت عُمان في نشر اإلسالم في شرق أفريقيا والهند وجنوب شرق آسيا عن طريق تجارها وبحّارتها. الدولة اليعربية والنهضة البحرية في القرن السابع عشر، برزت الدولة اليعربية كقوة بحرية عُمانية استطاعت طرد البرتغاليين من سوا

# Define Output Schema

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Dict, Literal


# --- Property choices ---
DifficultyLevel = Literal["Very Easy", "Easy", "Average", "Difficult"]
# BloomTaxonomy = Literal["Remember", "Apply", "Evaluate"]
# LanguageType = Literal["Arabic", "English"]
# ResponseTime = Literal["Short", "Medium", "Long"]



# --- Properties for a single question ---
class QuestionProperties(BaseModel):
    difficulty: DifficultyLevel = Field(..., alias="Difficulty")
    # bloom_taxonomy: BloomTaxonomy = Field(..., alias="Bloom Taxonomy")
    # language: LanguageType = Field(..., alias="Language")
    # response_time: ResponseTime = Field(..., alias="Response Time")


class Question(BaseModel):
    question_text: str = Field(..., description="Question text.")
    question_answers: List[str] = Field(..., min_items=2, description="List of answer options.")
    correct_answer: str = Field(..., description="The correct answer.")
    properties: QuestionProperties = Field(..., description="Per-question attributes.")


# --- Full question set schema ---
class QuestionSet(BaseModel):
    questions: List[Question] = Field(..., description="List of MCQs")


#Generate Dynamic **Prompt**

In [ ]:
def generate_prompt(number_of_questions, question_type, number_of_choices, properties, language):
    QUESTION_TYPE_MAP = {
        0: "multiple-choice questions (one correct answer)",
        1: "multiple-choice questions (multiple correct answers)",
        2: "true/false questions",
        3: "essay questions"
    }

    property_distributions = {}
    question_properties_lines = []
    total_questions_assigned = 0
    processed_items = []

    # First pass: calculate question counts from percentages and collect fixed values
    for prop_name, prop_value in properties.items():
        if isinstance(prop_value, list) and all(isinstance(item, dict) for item in prop_value):
            dist_parts = []
            enum_parts = []
            for item in prop_value:
                for k, v in item.items():
                    if isinstance(v, (int, float)):
                        count = int((v * number_of_questions) / 100)
                        processed_items.append((prop_name, k, v, count))
                        total_questions_assigned += count
                    else:
                        # Handle non-percentage items if necessary, although the current logic assumes percentage for lists of dicts
                         pass
            if dist_parts:
                property_distributions[prop_name] = ", ".join(dist_parts)
                question_properties_lines.append(
                    f'       "{prop_name}": {" | ".join([k.strip("") for k in enum_parts])}'
                )
        else:
            # Fixed value
            property_distributions[prop_name] = str(prop_value)
            question_properties_lines.append(f'       "{prop_name}": "{prop_value}"')


    # Adjust remainder to ensure total = number_of_questions for percentage-based distributions
    remainder = number_of_questions - total_questions_assigned
    if processed_items and remainder > 0:
        # Find the item with the largest initial count to add the remainder to
        processed_items.sort(key=lambda x: x[3], reverse=True)
        prop_name, k, v, count = processed_items[0]
        processed_items[0] = (prop_name, k, v, count + remainder)

    # Rebuild distribution strings and property lines based on adjusted counts
    property_distributions = {}
    question_properties_lines = []

    for prop_name, prop_value in properties.items():
         if isinstance(prop_value, list) and all(isinstance(item, dict) for item in prop_value):
            dist_parts = []
            enum_parts = []
            for item_prop_name, k, v, count in processed_items:
                if item_prop_name == prop_name:
                    if count > 0:
                         dist_parts.append(f'{k} ({count})')
                         enum_parts.append(f'"{k}"')
            if dist_parts:
                property_distributions[prop_name] = ", ".join(dist_parts)
                question_properties_lines.append(
                    f'       "{prop_name}": {" | ".join([k.strip("") for k in enum_parts])}'
                )
         else:
            property_distributions[prop_name] = str(prop_value)
            question_properties_lines.append(f'       "{prop_name}": "{prop_value}"')


    # Section for distribution description
    distribution_lines = "\n".join([
        f"   - {prop}: {desc}" for prop, desc in property_distributions.items() if prop != "extra_properties"
    ])

    # Properties block inside each question
    question_properties_block = ",\n".join(question_properties_lines)
    question_type_str = QUESTION_TYPE_MAP.get(question_type, "questions")

    # Final prompt
    prompt = f"""
You are a question generation expert.
You will be provided by a PDF content associated with a Pydantic scheme.,
generate exactly {number_of_questions} {question_type_str}.
Follow these instructions precisely:

1. Distribute the {number_of_questions} questions according to these constraints:
{distribution_lines}
follow the distribution proportions strictly.

2. Language of the output must be in {language}

⚠️ Do NOT include any explanation, formatting, or text outside the JSON object.
⚠️ Ensure the distribution rules are strictly followed. Extract content accurately and comply with the format.
""".strip()

    return prompt

In [ ]:

user_prompt = generate_prompt(number_of_questions = 5,question_type=0,number_of_choices=3,language='Arabic',properties={
    "Difficulty":[{"Easy":30,"Average":40,"Difficult":30}],
  })

In [ ]:
user_prompt

'You are a question generation expert.\nYou will be provided by a PDF content associated with a Pydantic scheme.,\ngenerate exactly 5 multiple-choice questions (one correct answer).\nFollow these instructions precisely:\n\n1. Distribute the 5 questions according to these constraints:\n   - Difficulty: Average (3), Easy (1), Difficult (1)\nfollow the distribution proportions strictly.\n\n2. Language of the output must be in Arabic\n\n⚠️ Do NOT include any explanation, formatting, or text outside the JSON object.\n⚠️ Ensure the distribution rules are strictly followed. Extract content accurately and comply with the format.'

In [ ]:
import json
messages = [
    {"role": "system","content":user_prompt},
    {
        "role": "user",
        "content": "\n".join([
            "## PDF content:",
            pdf_content.strip(),
            "",

            "## Pydantic Details:",
            json.dumps(
                QuestionSet.model_json_schema(), ensure_ascii=False
            ),
            "",

            "## Generated Questions:",
            "```json"
        ])
    }
]

In [ ]:
from openai import OpenAI
client  = OpenAI(api_key=openai_key)

In [ ]:
client

# Evaluation With Chatgpt 4o-mini Model

In [ ]:
model = "gpt-4o-mini"

chatgpt_response = client.chat.completions.create(model = model,messages = messages)

In [ ]:
chatgpt_response = chatgpt_response.choices[0].message.content.strip()

In [ ]:
json.loads(chatgpt_response)

{'questions': [{'question_text': 'ما هو الموقع الجغرافي لسلطنة عُمان؟',
   'question_answers': ['في الركن الشمالي الغربي من شبه الجزيرة العربية',
    'في الركن الجنوبي الشرقي من شبه الجزيرة العربية',
    'في وسط شبه الجزيرة العربية',
    'في الركن الجنوبي الغربي من شبه الجزيرة العربية'],
   'correct_answer': 'في الركن الجنوبي الشرقي من شبه الجزيرة العربية',
   'properties': {'Difficulty': 'Easy'}},
  {'question_text': 'متى دخل الإسلام إلى سلطنة عُمان؟',
   'question_answers': ['في القرن الثامن الميلادي',
    'في القرن السابع الميلادي',
    'في القرن السادس الميلادي',
    'في القرن التاسع الميلادي'],
   'correct_answer': 'في القرن السابع الميلادي',
   'properties': {'Difficulty': 'Average'}},
  {'question_text': 'ما هو المذهب الرئيسي في سلطنة عُمان؟',
   'question_answers': ['السني', 'الإباضي', 'الشيعي', 'الوهابي'],
   'correct_answer': 'الإباضي',
   'properties': {'Difficulty': 'Average'}},
  {'question_text': 'أي من القلاع التالية تُعتبر من أبرز رموز الحكم والدفاع في عُمان؟',
   'ques

# Evaluation With Base Model **Qwen**

In [ ]:
torch_dtype = None
base_model = AutoModelForCausalLM.from_pretrained(
    gbase_model_id,
    device_map = "auto",
    torch_dtype = torch_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(gbase_model_id)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text], return_tensors="pt").to(device)

generated_ids = base_model.generate(
    model_inputs.input_ids,
    max_new_tokens=1024,
    do_sample=False, top_k=None, temperature=None, top_p=None,
)

generated_ids = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [ ]:
json.loads(response)

[{'question_text': 'عُمان تُعد واحدة من أقدم الدول التي استوطنتها الحضارات البشرية، كيف يمكن أن نحدد هذا التاريخ؟',
  'question_answers': ['منذ حوالي 10,000 سنة',
   'منذ حوالي 8,000 سنة',
   'منذ حوالي 6,000 سنة',
   'منذ حوالي 4,000 سنة'],
  'correct_answer': 'منذ حوالي 10,000 سنة',
  'properties': {}},
 {'question_text': 'عُمان تُعتبر مركزًا للتجارة والمالحة منذ آلف السنين، كيف يمكن أن نفهم هذا التاريخ؟',
  'question_answers': ['منذ القرن الأول الميلادي',
   'منذ القرن الثاني الميلادي',
   'منذ القرن الثالث الميلادي',
   'منذ القرن الرابع الميلادي'],
  'correct_answer': 'منذ القرن الرابع الميلادي',
  'properties': {}},
 {'question_text': 'عُمان عبر العصور عُمان القديمة، كيف يمكن أن نفهم هذا التاريخ؟',
  'question_answers': ['عبر العصور الإسلامية',
   'عبر العصور الأسلامية',
   'عبر العصور الإسلامية واليهودية',
   'عبر العصور الإسلامية واليهودية والبوذية'],
  'correct_answer': 'عبر العصور الإسلامية',
  'properties': {}},
 {'question_text': 'عُمان تُعتبر مركزًا تجاريًا وسياسيًا تابعًا

## Evaluate With Gemma1:3b

In [ ]:
from transformers import Gemma3ForCausalLM , AutoTokenizer , BitsAndBytesConfig

gemma_model_id = "google/gemma-3-1b-it"

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

gemma_model = Gemma3ForCausalLM.from_pretrained(
    gemma_model_id, quantization_config=quantization_config
).eval()

gemma_tokenizer = AutoTokenizer.from_pretrained(gemma_model_id)


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [ ]:
device = "cuda"

text = gemma_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = gemma_tokenizer([text], return_tensors="pt").to(device)

generated_ids = gemma_model.generate(
    model_inputs.input_ids,
    max_new_tokens=1024,
    do_sample=False, top_k=None, temperature=None, top_p=None,
)

generated_ids = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = gemma_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [ ]:

response

'```json\n{\n  "questions": [\n    {\n      "question_text": "ما هي أهمية موقع سلطنة عُمان الجغرافي الفريد؟",\n      "question_answers": "تعتبر عُمان موقعًا استراتيجيًا على ممرات بحرية حيوية، مما ساهم في نموها كمركز تجاري ومالحي منذ آلاف السنين.",\n      "correct_answer": "موقعها الجغرافي الفريد، على ممرات بحرية حيوية.",\n      "properties": {\n        "Difficulty": "Average"\n      }\n    },\n    {\n      "question_text": "ما هي أبرز التراث المعماري والحضاري في عُمان؟",\n      "question_answers": "تزخر عُمان بتراثها المعماري والحضاري، بما في ذلك قلعة نزوى، وقلعة بهالء، وحصن جبرين، وغيرها.",\n      "correct_answer": "قلعة نزوى، وقلعة بهالء، وحصن جبرين.",\n      "properties": {\n        "Difficulty": "Average"\n      }\n    },\n    {\n      "question_text": "ما هي أهمية الحرف اليدوية والفنون التقليدية في عُمان؟",\n      "question_answers": "تُمارس الحرف اليدوية والفنون التقليدية مثل صناعة الخناجر، والسفن الخشبية، والنسيج، وغيرها، وهي جزء أساسي من الهوية الثقافية والوطنية.",\n      "corr

In [ ]:
from json_repair import repair_json
json.loads(repair_json(response))

{'questions': [{'question_text': 'ما هي أهمية موقع سلطنة عُمان الجغرافي الفريد؟',
   'question_answers': 'تعتبر عُمان موقعًا استراتيجيًا على ممرات بحرية حيوية، مما ساهم في نموها كمركز تجاري ومالحي منذ آلاف السنين.',
   'correct_answer': 'موقعها الجغرافي الفريد، على ممرات بحرية حيوية.',
   'properties': {'Difficulty': 'Average'}},
  {'question_text': 'ما هي أبرز التراث المعماري والحضاري في عُمان؟',
   'question_answers': 'تزخر عُمان بتراثها المعماري والحضاري، بما في ذلك قلعة نزوى، وقلعة بهالء، وحصن جبرين، وغيرها.',
   'correct_answer': 'قلعة نزوى، وقلعة بهالء، وحصن جبرين.',
   'properties': {'Difficulty': 'Average'}},
  {'question_text': 'ما هي أهمية الحرف اليدوية والفنون التقليدية في عُمان؟',
   'question_answers': 'تُمارس الحرف اليدوية والفنون التقليدية مثل صناعة الخناجر، والسفن الخشبية، والنسيج، وغيرها، وهي جزء أساسي من الهوية الثقافية والوطنية.',
   'correct_answer': 'صناعة الخناجر، والسفن الخشبية، والنسيج.',
   'properties': {'Difficulty': 'Average'}},
  {'question_text': 'ما هي أه

In [ ]:
json.dumps(QuestionSet.model_json_schema(), ensure_ascii=False),

('{"$defs": {"Question": {"properties": {"question_text": {"description": "Question text.", "title": "Question Text", "type": "string"}, "question_answers": {"description": "List of answer options.", "items": {"type": "string"}, "minItems": 2, "title": "Question Answers", "type": "array"}, "correct_answer": {"description": "The correct answer.", "title": "Correct Answer", "type": "string"}, "properties": {"$ref": "#/$defs/QuestionProperties", "description": "Per-question attributes."}}, "required": ["question_text", "question_answers", "correct_answer", "properties"], "title": "Question", "type": "object"}, "QuestionProperties": {"properties": {"Difficulty": {"enum": ["Very Easy", "Easy", "Average", "Difficult"], "title": "Difficulty", "type": "string"}}, "required": ["Difficulty"], "title": "QuestionProperties", "type": "object"}}, "properties": {"questions": {"description": "List of MCQs", "items": {"$ref": "#/$defs/Question"}, "title": "Questions", "type": "array"}}, "required": ["q